# 6 — Streamlit App: Design Notes

**Stage:** `data/clean/catalog_mood.csv` → `app/app.py`

## Purpose
Column selection and filter logic are worked out here; the **production
app lives in `app/app.py`**, not in this notebook.

**Conventions:** 🎯 goal · 🔍 observation · ✅ decision · ⚠️ limitation

## 1. What the app shows and how

### 1.1 Filters, selectors and eligibility flags

* Filters can be on: rating, publication year, number of pages. 
* Selection-tick box can be done by: List name, Book source
* Actual book groups recommendations can be done on two eligibility flags. Two different questions, not to be collapsed.

| Flag | Question it answers | App behaviour when `False` |
|---|---|---|
| `clusterable` | Was this book eligible for genre clustering? | Exclude from "more like this" recommendations |
| `mood_scored` | Did this book have a usable synopsis to score? | Exclude from the mood filter |

A book can be `clusterable=True, mood_scored=False`, or the reverse. Branch on each flag
**explicitly** — never infer eligibility from a null cluster or a null mood.

## 2. Load the final catalogue

🎯 `catalog_mood.csv` is the single output of the whole pipeline — cluster labels (notebook 4) and mood labels (notebook 5) already attached.

In [9]:
import pandas as pd

df = pd.read_csv("../data/clean/catalog_mood.csv")
print(df.info())
display(df.head(1))


<class 'pandas.DataFrame'>
RangeIndex: 1077 entries, 0 to 1076
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   book_name          1077 non-null   str    
 1   author             1077 non-null   str    
 2   book_url           1077 non-null   str    
 3   img_url            1077 non-null   str    
 4   rating             1077 non-null   float64
 5   pages              1077 non-null   float64
 6   pub_date           1077 non-null   int64  
 7   genres             1077 non-null   str    
 8   genres_clean       1077 non-null   str    
 9   synopsis           1077 non-null   str    
 10  data_source        1077 non-null   str    
 11  source_list        1077 non-null   str    
 12  cluster_f1b        612 non-null    float64
 13  cluster_name       612 non-null    str    
 14  clusterable        1077 non-null   bool   
 15  mood_scored        1077 non-null   bool   
 16  mood               1055 non-null   

,book_name,author,book_url,img_url,rating,pages,pub_date,genres,genres_clean,synopsis,...,emotion_secondary,emotion_conf,mood_low_conf,emo_anger,emo_disgust,emo_fear,emo_joy,emo_neutral,emo_sadness,emo_surprise
0,Persuasion,Jane Austen,https://play.google.com/store/books/details?id...,http://books.google.com/books/content?id=UrrTE...,4.12,260.0,2026,Fiction,['fiction'],Discover the timeless allure of love and secon...,...,joy,0.66172,False,0.004315,0.01351,0.003206,0.285717,0.66172,0.02295,0.008581


In [10]:
df.columns

Index(['book_name', 'author', 'book_url', 'img_url', 'rating', 'pages',
       'pub_date', 'genres', 'genres_clean', 'synopsis', 'data_source',
       'source_list', 'cluster_f1b', 'cluster_name', 'clusterable',
       'mood_scored', 'mood', 'mood_secondary', 'emotion', 'emotion_secondary',
       'emotion_conf', 'mood_low_conf', 'emo_anger', 'emo_disgust', 'emo_fear',
       'emo_joy', 'emo_neutral', 'emo_sadness', 'emo_surprise'],
      dtype='str')

In [13]:
df.source_list.unique()

<ArrowStringArray>
[                              'Best Books Ever',
                       '20th Century Literature',
                                   'Young Adult',
                         'Books You Should Read',
               'The Best Epic Fantasy (fiction)',
                             'Best Epic Fantasy',
         'Books That Should Be Made Into Movies',
                          'Best Book Boyfriends',
                        'Best Young Adult Books',
   'Best Dystopian and Post-Apocalyptic Fiction',
                   'Books That Should Be Movies',
                       'Best Historical Fiction',
 'Books That Everyone Should Read At Least Once',
               'Best Books of the Decade: 2000s',
                               'featured Google',
                'Best Books of the 20th Century']
Length: 16, dtype: str

## 3. Column selection for the app

🎯 The app does not need all 29 columns. Select only what the UI renders or filters on — everything else is pipeline bookkeeping.

In [11]:
app_cols= ['book_name', 'author', 'book_url', 'img_url', 'rating', 'pages',
       'pub_date','genres_clean', 'synopsis', 'data_source',
       'source_list', 'cluster_f1b', 'cluster_name', 'clusterable',
       'mood_scored', 'mood', 'mood_secondary', 'emotion', 'emotion_secondary',
       'emotion_conf', 'mood_low_conf']

app_df=df.copy()
app_df= app_df[app_cols]
app_df.head(1)

,book_name,author,book_url,img_url,rating,pages,pub_date,genres_clean,synopsis,data_source,...,cluster_f1b,cluster_name,clusterable,mood_scored,mood,mood_secondary,emotion,emotion_secondary,emotion_conf,mood_low_conf
0,Persuasion,Jane Austen,https://play.google.com/store/books/details?id...,http://books.google.com/books/content?id=UrrTE...,4.12,260.0,2026,['fiction'],Discover the timeless allure of love and secon...,google_books,...,NaN,NaN,False,True,Contemplative,Uplifting,neutral,joy,0.66172,False
